In [ ]:
import numpy as np
import traceback

In [ ]:
# Функция проверки
def check_lin_eq(gauss_lin_equations):
    examples = [
        {
            "name": "2x2, простой",
            "augmented": np.array([
                [1.0,  1.0, 3.0],
                [2.0, -1.0, 0.0],
            ], dtype=float),
            "solution": np.array([1.0, 2.0], dtype=float),
        },
        {
            "name": "3x3, единственное решение",
            "augmented": np.array([
                [1.0,  1.0,  1.0, 6.0],
                [2.0, -1.0,  1.0, 3.0],
                [1.0,  2.0, -1.0, 2.0],
            ], dtype=float),
            "solution": np.array([1.0, 2.0, 3.0], dtype=float),
        },
        {
            "name": "3x3, нужен обмен строк: первый ведущий элемент = 0",
            "augmented": np.array([
                [0.0, 1.0,  1.0, 3.0],
                [1.0, 1.0,  1.0, 4.0],
                [2.0, 1.0, -1.0, 1.0],
            ], dtype=float),
            "solution": np.array([1.0, 1.0, 2.0], dtype=float),
        },
        {
            "name": "3x3, решение с отрицательным числом",
            "augmented": np.array([
                [2.0,  1.0, -1.0, -1.0],
                [1.0, -1.0,  2.0, -1.0],
                [3.0,  2.0,  1.0,  2.0],
            ], dtype=float),
            "solution": np.array([-1.0, 2.0, 1.0], dtype=float),
        },
        {
            "name": "4x4, единственное решение",
            "augmented": np.array([
                [1.0,  1.0,  1.0,  1.0, 10.0],
                [2.0, -1.0,  1.0, -1.0, -1.0],
                [1.0,  2.0, -1.0,  1.0,  6.0],
                [3.0,  1.0,  2.0, -1.0,  7.0],
            ], dtype=float),
            "solution": np.array([1.0, 2.0, 3.0, 4.0], dtype=float),
        },
        {
            "name": "3x3, несовместная (третья строка противоречит сумме первых двух)",
            "kind": "inconsistent",
            "augmented": np.array([
                [1.0,  1.0, 1.0, 1.0],
                [1.0, -1.0, 1.0, 2.0],
                [2.0,  0.0, 2.0, 4.0],
            ], dtype=float),
            "solution": None,
        },
        {
            "name": "2x2, бесконечно много решений (1 свободная переменная)",
            "kind": "infinite",
            "augmented": np.array([
                [1.0, 1.0, 2.0],
                [2.0, 2.0, 4.0],
            ], dtype=float),
        }
    ]
    ok = True
    for example in examples:
        if 'kind' in example:
            try:
                gauss_lin_equations(example['augmented'])
            except Exception as e:
                pass
            else:
                ok = False
                print(f'Система, которая не имеет решений, должна выдавать ошибку!')
        else:
            try:
                solution = gauss_lin_equations(example['augmented'])
                if not np.isclose(solution.all(), example['solution'].all(), atol=1e-3):
                    ok = False
                    print(f'Неправильное решение системы {example["augmented"]}')
                    print(f'Ожидаемое решение: {example["solution"]}')
            except Exception as e:
                traceback.print_exc()
    if ok:
        print('Всё верно!')

# Решение систем линейных уравнений

Пусть необходимо решить систему линейных уравнений:

\begin{equation*}
    \begin{cases}
        a_{11} x_1 + a_{12} x_2 + ... + a_{1n} x_n  = b_1 \\
        a_{21} x_1 + a_{22} x_2 + ... + a_{2n} x_n  = b_2\\
        ...\\
        a_{n1} x_1 + a_{n2} x_2 + ... + a_{nn} x_n  = b_n
    \end{cases}
\end{equation*}

Из курса линейной алгебры Вам известен метод Крамера. Для решения системы, состоящей из $n$ корней и имеющей $n$ неизвестных, необходимо найти главный детерминант системы и $n$ вспомогательных. Итого, нужно сосчитать $n+1$ детерминант $n$-го порядка. Для нахождения каждого детерминанта $n$-го порядка необходимо рассчитать $n$ детерминантов $n-1$-го порядка, и т.д. Для $n>4$ метод Крамера требует слишком большого количества вычислений, поэтому целесообразно использование численных методов.

## Метод Гаусса

Для решения создадим ***расширенную матрицу*** системы: она включает главную матрицу $A_{n \times n}$ системы и столбец свободных членов $B_{n \times 1}$:

\begin{equation*}
    C_{n \times n+1} = 
    \begin{bmatrix}
        a_{11} & a_{12} & a_{13} & \dots & a_{1n} & b_1 \\
        a_{21} & a_{22} & a_{23} & \dots & a_{2n} & b_2 \\
        a_{31} & a_{32} & a_{33} & \dots & a_{3n} & b_3 \\
        \vdots & \vdots & \vdots & \ddots & \vdots & \vdots \\
        a_{n1} & a_{n2} & a_{n3} & \dots & a_{nn} & b_n \\
    \end{bmatrix}
\end{equation*}

Элементы этой матрицы будем называть $c_{ij}$.

Решение системы линейных уравнений (ЛУ) методом Гаусса проходит в два этапа: **Прямой** и **Обратный** ход.

### Прямой ход

1.  Выбирается ведущий элемент &mdash; коэффициент при неизвестном $x_1$. Если уравнение имеет единственное решение, то хотя бы в одном из уравнений коэффициент $a_{i1}$ не равен нулю. Поскольку смена уравнений не приводит к изменению корней, можно переставить уравнение с ненулевым $a_{i1}$ на первую строку. **С точки зрения программирования**: мы должны найти такую строку матрицы, чтобы $c_{i1} \neq 0$. Эту строку необходимо поменять с первой строкой матрицы $C$.

    В случае, если такой строки не нашлось ($\forall i = \overline{1, n}: c_{i1} = 0$), система ЛУ не имеет единственного решения. В программе можно выдать ошибку:

    ```
    if ...:
        raise RuntimeError('Система не имеет единственного решения!')
    ```

2. Необходимо разделить первое уравнение на коэффициент $a_{11}$. Это действие также не изменит корней уравнения. **С точки зрения программирования**: необходимо поделить все элементы первой строки матрицы $C$ на величину $c_{11} \neq 0$. В таком случае коэффициент $c_{11}$ будет равен $1$.

3. Умножим первую строку на коэффициент $a_{21}$ и вычтем её из второй строки. В таком случае коэффициент $a_21$ станет равным нулю. Аналогично для всех остальных строк. **С точки зрения программирования**: для каждой $i$-й строки матрицы $C$, кроме $i=1$, умножаем первую строку матрицы на коэффициент $c_{i1}$ и вычитаем получившуюся строку из $i$-й строки. Получим матрицу вида:

    \begin{equation*}
        C'_{n \times n+1} = 
        \begin{bmatrix}
            1 & a_{12} & a_{13} & \dots & a_{1n} & b'_1 \\
            0 & a_{22} & a_{23} & \dots & a_{2n} & b'_2 \\
            0 & a_{32} & a_{33} & \dots & a_{3n} & b'_3 \\
            \vdots & \vdots & \vdots & \ddots & \vdots & \vdots \\
            0 & a_{n2} & a_{n3} & \dots & a_{nn} & b'_n \\
        \end{bmatrix}
    \end{equation*}

4. Повторяем пукты 1-3 для каждой последующей строки $j$ и ведущих коэффициентов $a_{jj}$. На каждом этапе необходимо работать только со строками, номера которых не меньше $j$ (верхнюю часть матрицы уже не меняем). В результате таких линейных преобразований получим матрицу $C$ вида:

    \begin{equation*}
        C'_{n \times n+1} = 
        \begin{bmatrix}
            1 & a'_{12} & a'_{13} & \dots & a'_{1n} & b'_1 \\
            0 & 1 & a'_{23} & \dots & a'_{2n} & b'_2 \\
            0 & 0 & 1 & \dots & a'_{3n} & b'_3 \\
            \vdots & \vdots & \vdots & \ddots & \vdots & \vdots \\
            0 & 0 & 0 & \dots & 1 & b'_n \\
        \end{bmatrix}
    \end{equation*}

    Коэффициенты $b'_i$ изменились в результате линейных преобразований соответствующим образом. В результате прямого хода мы преобразовали главную матрицу системы $A$ к треугольному виду (все её члены, находящиеся ниже основной диагонали, равны нулю) и соответствующим образом преобразовали столбец свободных членов.

    Если на каком-то шаге решения не удалось выполнить пункт 1 (не нашлось ведущего коэффициента $a_{jj} \neq 0$), то система не имеет единственного решения. Можно выдать ошибку.

### Обратный ход

1. Запишем систему, получившуюся после прямого хода:

    \begin{align*}
        x_1 + a'_{12} x_2 + a'_{13} x_3 + ... + a'_{1n} x_n &= b'_1 \\
        x_2 + a'_{23} x_3 + ... + a'_{2n} x_n &= b'_2\\
        x_3 + ... + a'_{3n} x_n &= b'_3\\
        ...\\
        x_n &= b'_n
    \end{align*}

    Из последнего уравнения сразу найдем $n-й$ корень: $x_n$ = $b'_n$. (В программе $x_n = c_{n,n+1}$)

2. Подставляя $x_n$ в предпоследнее уравнение, найдем $x_{n-1} = b'_{n-1} - a'_{n-1,n} \cdot x_n $

3. Аналогично находятся остальные корни уравнения.

### Программирование метода Гаусса

В ячейке ниже реализуйте функцию для решения системы линейных уравнений методом Гаусса. Аргумент функции &mdash; расширенная матрица $C_{n \times n+1}$. Функция должна возвращать массив корней $x_i$.

`Полезная информация:`

Для проведения линейных преобразований можно взять всю строку матрицы `numpy` так:

```
row = c[i]
```

Объект `row` &mdash; это массив `numpy`, содержащий все элементы $i$-й строки. Операции над строками делаются аналогично:

```
c[i] = new_row  # Заменили i-ю строку на новую
c[i] += new_row  # Прибавили строку к i-й строке
```

In [ ]:
def gauss_lin_equations(c: list[list[float]]) -> list[float]:
    """Решение системы линейных уравнений методом Гаусса.

    Args:
        c (array_like): Расширенная матрица n x n+1 системы уравнений.

    Returns:
        list[float]: Корни уравнения.
    """
    # TODO Проверьте размеры матрицы. Количество столбцов должно быть на 1 больше количества строк. Если это не так, поднимите ошибку (raise ...Error('...'))
    
    c = np.asarray(c)  # Перевели в массив numpy
    
    # TODO Реализуйте прямой ход метода гаусса, применяя линейные преобразования строк матрицы.
    
    # TODO Реализуйте обратный ход и найдите корни уравнения. Верните массив корней (индексы корней в массиве должны быть по возрастанию)
    
    return ...

# Проверяем
check_lin_eq(gauss_lin_equations)

## Итерационный метод Зейделя

Пусть все диагональные члены матрицы $A_{n \times n}$ не равны нулю. К такому виду матрицы можно прийти путем перестановки строк уравнения. Выразим из первого уравнения $x_1$, из второго &mdash; $x_2$, из третьего &mdash; $x_3$, и так далее. Получим следующую систему:

\begin{equation*}
    \begin{cases}
        x_1 =  d_1 + c_{12} x_2 + c_{13} x_3 ... + c_{1n} x_n \\
        x_2 =  d_2 + c_{21} x_1 + c_{23} x_3 ... + c_{2n} x_n \\
        ...\\
        x_n =  d_n + c_{n1} x_1 + c_{n2} x_2 ... + c_{nn} x_n \\
    \end{cases},
\end{equation*}

где

$$c_{ij} = - a_{ij} / a_{ii} \ \text{при} \ i \neq j,$$
$$c_{ii} = 0 \ \forall i = \overline{1, n},$$
$$d_i = b_i / a_{ii}.$$

Составим элементы $c_{ij}$ в матрицу $C_{n \times n}$, а элементы $d_i$ &mdash; в вектор-столбец $\vec d$. Тогда в матричной форме система имеет вид:

$$ \vec x = C \vec x + \vec d$$

Это выражение можно переписать как рекуррентное соотношение, в котором вектор-столбец корней $\vec x$ выражен сам через себя:

$$\vec x^{(k+1)} = C \vec x^{(k)} + \vec d$$

Применяя это выражение несколько раз, каждый раз находя новый набор корней на основе предыдущего, мы будем приближаться к истинным значениям корней уравнения.

### Условие сходимости решения

Рекуррентное соотношение, написанное выше, сходится к истинному значению $\vec x$ при многократном применении лишь при определенных условиях. Основным условием является следующее:

$$\forall i = \overline{1,n}: \ \left| a_{ii} \right| > \sum \limits_{j=1, j \neq i}^n \left| a_{ij} \right|$$

При этом метод Зейделя может сходиться и при других условиях, о которых можно прочитать в учебном пособии (стр. 80).

Условием остановки метода Зейделя является отличие всех корней уравнения от истинного значения не более, чем на заданную величину $\varepsilon$. Для этого достаточно выполнения условия:

$$\forall i = \overline{1,n}: \ \left|x_i^{(k)} - x_i^{(k-1)} \right| < \varepsilon$$

### Программирование метода Зейделя

Для реализации метода Зейделя необходимо выполнить следующие шаги:

1. Убедиться, что диагональные члены матрицы $A$ не равны нулю. При необходимости поменять уравнения так, чтобы это условие выполнилось.

2. Проверить условие сходимости метода Зейделя. Если оно не выполено, выдать ошибку. 

3. Найти матрицы $C$ и $\vec d$ согласно методу, описанному выше.

4. Реализовать итерации по методу Зейделя в цикле с условием остановки.

    Важной особенностью метода Зейделя является то, что на любом $k$-м шаге итерации каждый последующий корень уравнения вычисляется на основании только что рассчитанных корней $k$-го шага. 

    Например, если вычислено $k$-е приближение, то $(k+1)$-е приближение следует вычислять по формулам:

    $$x_1^{(k+1)} = d_1 + \sum \limits_{j=1}^n c_{1j} x_j^{(k)},$$

    $$x_2^{(k+1)} = d_2 + c_{21} x_1^{(k+1)} + \sum \limits_{j=2}^n c_{2j} x_j^{(k)},$$

    $$...$$

    $$x_n^{(k+1)} = d_n + \sum \limits_{j=1}^{n-1} c_{nj} x_j^{(k+1)} + c_{nn} x_n^{(k)}$$

In [ ]:
def seidel_lin_equations(matrix: list[list[float]]) -> list[float]:
    """Решение системы линейных уравнений методом Зейделя.

    Args:
        matrix (array_like): Расширенная матрица n x n+1 системы уравнений.

    Returns:
        list[float]: Корни уравнения.
    """
    # TODO Проверьте размеры матрицы. Количество столбцов должно быть на 1 больше количества строк. Если это не так, поднимите ошибку (raise ...Error('...'))
    
    matrix = np.asarray(matrix)  # Перевели в массив numpy
    
    # TODO Разбейте расширенную матрицу на матрицу A и вектор-столбец b.
    
    # TODO Проверьте, чтобы диагональные элементы матрицы A не были нулевыми. При необходимости замените строки. 
    # Если меняете строки матрицы A, не забудьте заменить их в векторе b!
    
    # TODO Проверьте условие сходимости метода Зейделя. Если оно не выполняется, поднимите ошибку.
    
    # TODO Найдите матрицы C и d
    
    # TODO Реализуйте итерации метода Зейделя.
    
    return ...

### Проверка метода Зейделя

Для проверки метода Зейделя решите систему линейных уравнений:

\begin{equation*}
    \begin{cases}
        9 x_1 - 2 x_2 + x_3 = 8 \\
        2 x_1 - 7 x_2 + x_3 = -4 \\
        x_1 + 3 x_2 + 8 x_3 = 12
    \end{cases}
\end{equation*}

Ожидаемые корни уравнения:

$$\vec x = \left( 1, \ 1, \ 1\right)$$